# Base de Dados Financeira - Versão em Português

Este notebook carrega todas as tabelas da base de dados financeira do BIRD benchmark com os nomes das colunas traduzidos para português brasileiro.

## Requisitos

### 1. Download da Base de Dados

Faça o download da base de dados MiniDev do repositório oficial:
- **Repositório:** https://github.com/bird-bench/mini_dev
- **Arquivo necessário:** `financial.sqlite` localizado em `dev_databases/financial/`

### 2. Estrutura de Pastas Esperada

O notebook espera que o arquivo `financial.sqlite` esteja no seguinte caminho relativo:
```
baseMiniDev/minidev/MINIDEV/dev_databases/financial/financial.sqlite
```

### 3. Bibliotecas Python Necessárias

Instale as seguintes bibliotecas caso ainda não as tenha:

```bash
pip install pandas deep-translator
```

Notas:
- `sqlite3` geralmente já vem incluído na instalação padrão do Python.
- `deep-translator` é uma biblioteca robusta para tradução automática que suporta múltiplos serviços (Google Translate, Microsoft, etc.).

## Sobre a Base de Dados

Esta base de dados contém informações financeiras de um banco tcheco, incluindo:
- **account**: Contas bancárias
- **card**: Cartões de crédito
- **client**: Clientes do banco
- **disp**: Disposições (relacionamento cliente-conta)
- **district**: Informações demográficas dos distritos
- **loan**: Empréstimos concedidos

- **order**: Ordens de pagamento permanentes- **trans**: Transações bancárias

## Importação de Bibliotecas

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path
from deep_translator import GoogleTranslator
import unicodedata

## Função de Tradução Automática

Utilizamos a biblioteca `deep-translator` para traduzir automaticamente os valores das colunas do tcheco para o português.

In [2]:
# Inicializar tradutor com detecção automática para português
tradutor = GoogleTranslator(source='auto', target='pt')

def remover_acentos(texto):
    """
    Remove acentos e caracteres especiais de um texto.
    
    Parâmetros:
    - texto: string para normalizar
    
    Retorna:
    - string sem acentos
    """
    if pd.isna(texto):
        return texto
    # Normaliza o texto (NFD = Normalization Form Decomposed)
    # e remove marcas diacríticas (acentos)
    nfkd = unicodedata.normalize('NFD', str(texto))
    texto_sem_acento = ''.join([c for c in nfkd if not unicodedata.combining(c)])
    return texto_sem_acento

def traduzir_valores_coluna(serie, valores_unicos_limite=50):
    """
    Traduz os valores únicos de uma Series do pandas usando deep-translator.
    Remove acentos e caracteres especiais das traduções.
    
    Parâmetros:
    - serie: pd.Series com valores a serem traduzidos
    - valores_unicos_limite: limite de valores únicos para traduzir (segurança)
    
    Retorna:
    - pd.Series com valores traduzidos e sem acentos
    """
    # Obter valores únicos não nulos
    valores_unicos = serie.dropna().unique()
    
    # Verificar se o número de valores únicos é razoável
    if len(valores_unicos) > valores_unicos_limite:
        print(f"Aviso: Muitos valores únicos ({len(valores_unicos)}). Pulando tradução.")
        return serie
    
    # Criar dicionário de tradução
    dicionario_traducao = {}
    for valor in valores_unicos:
        try:
            # Traduzir o valor
            traducao = tradutor.translate(str(valor))
            # Remover acentos e converter para maiúsculas
            traducao_sem_acento = remover_acentos(traducao).upper()
            dicionario_traducao[valor] = traducao_sem_acento
        except Exception as e:
            print(f"Erro ao traduzir '{valor}': {e}")
            dicionario_traducao[valor] = valor  # Manter original em caso de erro
    
    # Aplicar tradução
    return serie.replace(dicionario_traducao)

print("Função de tradução automática configurada com sucesso.")

Função de tradução automática configurada com sucesso.


## Configuração do Caminho da Base de Dados

In [ ]:
# Defina o caminho para o arquivo SQLite
#db_path = Path('baseMiniDev/minidev/MINIDEV/dev_databases/financial/financial.sqlite')
db_path = Path('mini-dev/pt-br/financial.sqlite')
# Verificar se o arquivo existe
if not db_path.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {db_path}\n"
        "Por favor, faça o download da base de dados de: https://github.com/bird-bench/mini_dev"
    )

# Criar conexão com o banco de dados
conn = sqlite3.connect(db_path)
print(f"Conexão estabelecida com: {db_path}")

Conexão estabelecida com: mini-dev/original/financial.sqlite


## Função de Tradução de Nomes de Colunas

Função auxiliar para traduzir automaticamente os nomes das colunas do inglês para o português.

In [4]:
# Inicializar tradutor do inglês para português para os nomes das colunas
tradutor_colunas = GoogleTranslator(source='en', target='pt')

def traduzir_nomes_colunas(df):
    """
    Traduz os nomes das colunas de um DataFrame do inglês para português.
    Remove acentos e caracteres especiais dos nomes traduzidos.
    
    Parâmetros:
    - df: DataFrame do pandas
    
    Retorna:
    - DataFrame com nomes de colunas traduzidos e sem acentos
    """
    dicionario_colunas = {}
    for coluna in df.columns:
        try:
            # Traduzir nome da coluna
            # Substituir underscores por espaços para melhor tradução
            nome_formatado = coluna.replace('_', ' ')
            traducao = tradutor_colunas.translate(nome_formatado)
            # Remover acentos da tradução
            traducao_sem_acento = remover_acentos(traducao)
            # Formatar tradução: remover espaços e usar snake_case
            nome_traduzido = traducao_sem_acento.lower().replace(' ', '_')
            dicionario_colunas[coluna] = nome_traduzido
        except Exception as e:
            print(f"Erro ao traduzir coluna '{coluna}': {e}")
            dicionario_colunas[coluna] = coluna.lower()
    
    return df.rename(columns=dicionario_colunas)

print('Função de tradução de colunas configurada com sucesso.')

Função de tradução de colunas configurada com sucesso.


# Tabela 1: Account (Contas Bancárias)

Contém informações sobre as contas bancárias, incluindo localização da agência, periodicidade de emissão de extratos e data de abertura.

In [5]:
# Importar tabela account
df_conta = pd.read_sql_query("SELECT * FROM account", conn)
df_conta = traduzir_nomes_colunas(df_conta)

print(f"Tabela 'Account' carregada: {df_conta.shape[0]} registros, {df_conta.shape[1]} colunas")
print(f"Colunas: {', '.join(df_conta.columns)}")

Tabela 'Account' carregada: 4500 registros, 4 colunas
Colunas: id_da_conta, id_do_distrito, frequencia, data


## Visualização ANTES da tradução

In [6]:
# Visualização ANTES da tradução
colunas_freq = [col for col in df_conta.columns if 'freq' in col.lower() or 'period' in col.lower()]
if colunas_freq:
    coluna_frequencia = colunas_freq[0]
    print(f"Valores ORIGINAIS na coluna '{coluna_frequencia}':")
    print(df_conta[coluna_frequencia].value_counts())
    print()

print("Primeiras linhas ANTES da tradução:")
df_conta.head()

Valores ORIGINAIS na coluna 'frequencia':
frequencia
POPLATEK MESICNE      4167
POPLATEK TYDNE         240
POPLATEK PO OBRATU      93
Name: count, dtype: int64

Primeiras linhas ANTES da tradução:


,id_da_conta,id_do_distrito,frequencia,data
0,1,18,POPLATEK MESICNE,1995-03-24
1,2,1,POPLATEK MESICNE,1993-02-26
2,3,5,POPLATEK MESICNE,1997-07-07
3,4,12,POPLATEK MESICNE,1996-02-21
4,5,15,POPLATEK MESICNE,1997-05-30


## Tradução dos valores categóricos

In [7]:
# Traduzir valores categóricos
colunas_freq = [col for col in df_conta.columns if 'freq' in col.lower() or 'period' in col.lower()]
if colunas_freq:
    coluna_frequencia = colunas_freq[0]
    df_conta[coluna_frequencia] = traduzir_valores_coluna(df_conta[coluna_frequencia])
    print(f"Valores TRADUZIDOS na coluna '{coluna_frequencia}':")
    print(df_conta[coluna_frequencia].value_counts())
else:
    print("Coluna de frequência não encontrada.")

Valores TRADUZIDOS na coluna 'frequencia':
frequencia
TAXA MENSAL            4167
TAXA POR SEMANA         240
TAXA APOS O ROTEIRO      93
Name: count, dtype: int64


## Visualização DEPOIS da tradução

In [8]:
print("Primeiras linhas DEPOIS da tradução:")
df_conta.head()

Primeiras linhas DEPOIS da tradução:


,id_da_conta,id_do_distrito,frequencia,data
0,1,18,TAXA MENSAL,1995-03-24
1,2,1,TAXA MENSAL,1993-02-26
2,3,5,TAXA MENSAL,1997-07-07
3,4,12,TAXA MENSAL,1996-02-21
4,5,15,TAXA MENSAL,1997-05-30


# Tabela 2: Card (Cartões de Crédito)

Informações sobre cartões de crédito emitidos, incluindo tipo (junior, classic, gold) e data de emissão.

In [9]:
# Importar tabela card
df_cartao = pd.read_sql_query("SELECT * FROM card", conn)
df_cartao = traduzir_nomes_colunas(df_cartao)

print(f"Tabela 'Card' carregada: {df_cartao.shape[0]} registros, {df_cartao.shape[1]} colunas")
print(f"Colunas: {', '.join(df_cartao.columns)}")

Tabela 'Card' carregada: 892 registros, 4 colunas
Colunas: identificacao_do_cartao, id_de_disp., tipo, publicado


In [10]:
df_cartao.head()

,identificacao_do_cartao,id_de_disp.,tipo,publicado
0,1,9,gold,1998-10-16
1,2,19,classic,1998-03-13
2,3,41,gold,1995-09-03
3,4,42,classic,1998-11-26
4,5,51,junior,1995-04-24


# Tabela 3: Client (Clientes)

Dados cadastrais dos clientes do banco, incluindo informações demográficas básicas.

In [11]:
# Importar tabela client
df_cliente = pd.read_sql_query("SELECT * FROM client", conn)
df_cliente = traduzir_nomes_colunas(df_cliente)

print(f"Tabela 'Client' carregada: {df_cliente.shape[0]} registros, {df_cliente.shape[1]} colunas")
print(f"Colunas: {', '.join(df_cliente.columns)}")
df_cliente.head()

Tabela 'Client' carregada: 5369 registros, 4 colunas
Colunas: id_do_cliente, genero, data_de_nascimento, id_do_distrito


,id_do_cliente,genero,data_de_nascimento,id_do_distrito
0,1,F,1970-12-13,18
1,2,M,1945-02-04,1
2,3,F,1940-10-09,1
3,4,M,1956-12-01,5
4,5,F,1960-07-03,5


# Tabela 4: Disp (Disposições)

Relacionamento entre clientes e contas, indicando o tipo de disposição (proprietário, usuário ou disponente).

In [12]:
# Importar tabela disp
df_disposicao = pd.read_sql_query("SELECT * FROM disp", conn)
df_disposicao = traduzir_nomes_colunas(df_disposicao)

print(f"Tabela 'Disp' carregada: {df_disposicao.shape[0]} registros, {df_disposicao.shape[1]} colunas")
print(f"Colunas: {', '.join(df_disposicao.columns)}")

Tabela 'Disp' carregada: 5369 registros, 4 colunas
Colunas: id_de_disp., id_do_cliente, id_da_conta, tipo


## Visualização ANTES da tradução

In [13]:
# Visualização ANTES da tradução
colunas_tipo = [col for col in df_disposicao.columns if 'tipo' in col.lower() or 'type' in col.lower()]
if colunas_tipo:
    coluna_tipo = colunas_tipo[0]
    print(f"Valores ORIGINAIS na coluna '{coluna_tipo}':")
    print(df_disposicao[coluna_tipo].value_counts())
    print()

print("Primeiras linhas ANTES da tradução:")
df_disposicao.head()

Valores ORIGINAIS na coluna 'tipo':
tipo
OWNER        4500
DISPONENT     869
Name: count, dtype: int64

Primeiras linhas ANTES da tradução:


,id_de_disp.,id_do_cliente,id_da_conta,tipo
0,1,1,1,OWNER
1,2,2,2,OWNER
2,3,3,2,DISPONENT
3,4,4,3,OWNER
4,5,5,3,DISPONENT


## Tradução dos valores categóricos

In [14]:
# Traduzir valores categóricos
colunas_tipo = [col for col in df_disposicao.columns if 'tipo' in col.lower() or 'type' in col.lower()]
if colunas_tipo:
    coluna_tipo = colunas_tipo[0]
    df_disposicao[coluna_tipo] = traduzir_valores_coluna(df_disposicao[coluna_tipo])
    print(f"Valores TRADUZIDOS na coluna '{coluna_tipo}':")
    print(df_disposicao[coluna_tipo].value_counts())
else:
    print("Coluna de tipo não encontrada.")

Valores TRADUZIDOS na coluna 'tipo':
tipo
PROPRIETARIO    4500
DISTRIBUIDOR     869
Name: count, dtype: int64


## Visualização DEPOIS da tradução

In [15]:
print("Primeiras linhas DEPOIS da tradução:")
df_disposicao.head()

Primeiras linhas DEPOIS da tradução:


,id_de_disp.,id_do_cliente,id_da_conta,tipo
0,1,1,1,PROPRIETARIO
1,2,2,2,PROPRIETARIO
2,3,3,2,DISTRIBUIDOR
3,4,4,3,PROPRIETARIO
4,5,5,3,DISTRIBUIDOR


# Tabela 5: District (Distritos)

Informações demográficas e socioeconômicas dos distritos, incluindo dados populacionais, taxa de desemprego, salário médio e criminalidade.

In [16]:
# Importar tabela district
df_distrito = pd.read_sql_query("SELECT * FROM district", conn)
df_distrito = traduzir_nomes_colunas(df_distrito)

print(f"Tabela 'District' carregada: {df_distrito.shape[0]} registros, {df_distrito.shape[1]} colunas")
print(f"Colunas: {', '.join(df_distrito.columns)}")
print("\nPrimeiras linhas da tabela District:")
df_distrito.head()

Tabela 'District' carregada: 77 registros, 16 colunas
Colunas: id_do_distrito, a2, a3, a4, a5, a6, a7, a8, a9, a10, a11, a12, a13, a14, a15, a16

Primeiras linhas da tabela District:


,id_do_distrito,a2,a3,a4,a5,a6,a7,a8,a9,a10,a11,a12,a13,a14,a15,a16
0,1,Hl.m. Praha,Prague,1204953,0,0,0,1,1,100.0,12541,0.2,0.43,167,85677.0,99107
1,2,Benesov,central Bohemia,88884,80,26,6,2,5,46.7,8507,1.6,1.85,132,2159.0,2674
2,3,Beroun,central Bohemia,75232,55,26,4,1,5,41.7,8980,1.9,2.21,111,2824.0,2813
3,4,Kladno,central Bohemia,149893,63,29,6,2,6,67.4,9753,4.6,5.05,109,5244.0,5892
4,5,Kolin,central Bohemia,95616,65,30,4,1,6,51.4,9307,3.8,4.43,118,2616.0,3040


# Tabela 6: Loan (Empréstimos)

Registro de empréstimos concedidos, incluindo valor, prazo, parcelas mensais e status de pagamento.

In [17]:
# Importar tabela loan
df_emprestimo = pd.read_sql_query("SELECT * FROM loan", conn)
df_emprestimo = traduzir_nomes_colunas(df_emprestimo)

print(f"Tabela 'Loan' carregada: {df_emprestimo.shape[0]} registros, {df_emprestimo.shape[1]} colunas")
print(f"Colunas: {', '.join(df_emprestimo.columns)}")
df_emprestimo.head()

Tabela 'Loan' carregada: 682 registros, 7 colunas
Colunas: id_do_emprestimo, id_da_conta, data, quantia, duracao, pagamentos, status


,id_do_emprestimo,id_da_conta,data,quantia,duracao,pagamentos,status
0,4959,2,1994-01-05,80952,24,3373.0,A
1,4961,19,1996-04-29,30276,12,2523.0,B
2,4962,25,1997-12-08,30276,12,2523.0,A
3,4967,37,1998-10-14,318480,60,5308.0,D
4,4968,38,1998-04-19,110736,48,2307.0,C


# Tabela 7: Order (Ordens de Pagamento)

Ordens de pagamento permanentes configuradas nas contas, especificando beneficiário, valor e finalidade.

In [18]:
# Importar tabela order
df_ordem = pd.read_sql_query("SELECT * FROM `order`", conn)
df_ordem = traduzir_nomes_colunas(df_ordem)

# Substituir códigos conhecidos por descrições em inglês
# para depois traduzir automaticamente para português.
colunas_para_substituicao = [
    col for col in df_ordem.columns
    if ('value' in col.lower() and 'description' in col.lower())
    or ('valor' in col.lower() and 'descricao' in col.lower())
    or ('simbolo' in col.lower() or 'symbol' in col.lower())
]

if colunas_para_substituicao:
    mapeamento_codigos_order = {
        'POJISTNE': 'insurance payment',
        'SIPO': 'household payment',
        'LEASING': 'leasing',
        'UVER': 'loan payment'
    }
    for coluna in colunas_para_substituicao:
        df_ordem[coluna] = df_ordem[coluna].replace(mapeamento_codigos_order)
    print(f"Substituições aplicadas nas colunas: {', '.join(colunas_para_substituicao)}")
else:
    print("Nenhuma coluna alvo encontrada para substituição de códigos da tabela Order.")

print(f"Tabela 'Order' carregada: {df_ordem.shape[0]} registros, {df_ordem.shape[1]} colunas")
print(f"Colunas: {', '.join(df_ordem.columns)}")

Substituições aplicadas nas colunas: simbolo_k
Tabela 'Order' carregada: 6471 registros, 6 colunas
Colunas: id_do_pedido, id_da_conta, banco_para, conta_para, quantia, simbolo_k


In [19]:
df_ordem

,id_do_pedido,id_da_conta,banco_para,conta_para,quantia,simbolo_k
0,29401,1,YZ,87144583,2452.0,household payment
1,29402,2,ST,89597016,3372.7,loan payment
2,29403,2,QR,13943797,7266.0,household payment
3,29404,3,WX,83084338,1135.0,household payment
4,29405,3,CD,24485939,327.0,
...,...,...,...,...,...,...
6466,46334,11362,YZ,70641225,4780.0,household payment
6467,46335,11362,MN,78507822,56.0,
6468,46336,11362,ST,40799850,330.0,insurance payment
6469,46337,11362,KL,20009470,129.0,


## Visualização ANTES da tradução

In [20]:
# Visualização ANTES da tradução
colunas_simbolo = [col for col in df_ordem.columns if 'simbolo' in col.lower() or 'symbol' in col.lower()]
if colunas_simbolo:
    coluna_simbolo = colunas_simbolo[0]
    print(f"Valores ORIGINAIS na coluna '{coluna_simbolo}':")
    print(df_ordem[coluna_simbolo].value_counts())
    print()

print("Primeiras linhas ANTES da tradução:")
df_ordem.head()

Valores ORIGINAIS na coluna 'simbolo_k':
simbolo_k
household payment    3502
                     1379
loan payment          717
insurance payment     532
leasing               341
Name: count, dtype: int64

Primeiras linhas ANTES da tradução:


,id_do_pedido,id_da_conta,banco_para,conta_para,quantia,simbolo_k
0,29401,1,YZ,87144583,2452.0,household payment
1,29402,2,ST,89597016,3372.7,loan payment
2,29403,2,QR,13943797,7266.0,household payment
3,29404,3,WX,83084338,1135.0,household payment
4,29405,3,CD,24485939,327.0,


## Tradução dos valores categóricos

In [21]:
# Traduzir valores categóricos
colunas_simbolo = [col for col in df_ordem.columns if 'simbolo' in col.lower() or 'symbol' in col.lower()]
colunas_value_description = [
    col for col in df_ordem.columns
    if ('value' in col.lower() and 'description' in col.lower())
    or ('valor' in col.lower() and 'descricao' in col.lower())
]

if colunas_simbolo:
    coluna_simbolo = colunas_simbolo[0]
    df_ordem[coluna_simbolo] = traduzir_valores_coluna(df_ordem[coluna_simbolo])
    print(f"Valores TRADUZIDOS na coluna '{coluna_simbolo}':")
    print(df_ordem[coluna_simbolo].value_counts())
else:
    print("Coluna símbolo não encontrada.")

if colunas_value_description:
    coluna_value_description = colunas_value_description[0]
    df_ordem[coluna_value_description] = traduzir_valores_coluna(df_ordem[coluna_value_description])
    print(f"Valores TRADUZIDOS na coluna '{coluna_value_description}':")
    print(df_ordem[coluna_value_description].value_counts())
else:
    print("Coluna value_description (ou equivalente traduzida) não encontrada para tradução.")

# Fallback para garantir que nenhum código original permaneça sem tradução.
mapeamento_fallback_pt = {
    'POJISTNE': 'PAGAMENTO DE SEGURO',
    'SIPO': 'PAGAMENTO DOMESTICO',
    'LEASING': 'LEASING',
    'UVER': 'PAGAMENTO DE EMPRESTIMO'
}
for coluna in set(colunas_simbolo + colunas_value_description):
    df_ordem[coluna] = df_ordem[coluna].replace(mapeamento_fallback_pt)

Valores TRADUZIDOS na coluna 'simbolo_k':
simbolo_k
PAGAMENTO DOMESTICO        3502
                           1379
PAGAMENTO DO EMPRESTIMO     717
PAGAMENTO DE SEGURO         532
LOCACAO                     341
Name: count, dtype: int64
Coluna value_description (ou equivalente traduzida) não encontrada para tradução.


## Visualização DEPOIS da tradução

In [22]:
print("Primeiras linhas DEPOIS da tradução:")
df_ordem.head()

Primeiras linhas DEPOIS da tradução:


,id_do_pedido,id_da_conta,banco_para,conta_para,quantia,simbolo_k
0,29401,1,YZ,87144583,2452.0,PAGAMENTO DOMESTICO
1,29402,2,ST,89597016,3372.7,PAGAMENTO DO EMPRESTIMO
2,29403,2,QR,13943797,7266.0,PAGAMENTO DOMESTICO
3,29404,3,WX,83084338,1135.0,PAGAMENTO DOMESTICO
4,29405,3,CD,24485939,327.0,


# Tabela 8: Trans (Transações)

Registro detalhado de todas as transações bancárias, incluindo depósitos, saques, transferências e saldo resultante.

In [23]:
# Importar tabela trans
df_transacao = pd.read_sql_query("SELECT * FROM trans", conn)
df_transacao = traduzir_nomes_colunas(df_transacao)

print(f"Tabela 'Trans' carregada: {df_transacao.shape[0]} registros, {df_transacao.shape[1]} colunas")
print(f"Colunas: {', '.join(df_transacao.columns)}")

Tabela 'Trans' carregada: 1056320 registros, 10 colunas
Colunas: identificacao_trans, id_da_conta, data, tipo, operacao, quantia, equilibrio, simbolo_k, banco, conta


## Visualização ANTES da tradução

In [24]:
# Visualização ANTES da tradução - amostra limitada devido ao tamanho da tabela
colunas_tipo = [col for col in df_transacao.columns if 'tipo' in col.lower() or 'type' in col.lower()]
colunas_operacao = [col for col in df_transacao.columns if 'operacao' in col.lower() or 'operation' in col.lower()]
colunas_simbolo = [col for col in df_transacao.columns if 'simbolo' in col.lower() or 'symbol' in col.lower()]

if colunas_tipo:
    print(f"Valores ORIGINAIS na coluna '{colunas_tipo[0]}':")
    print(df_transacao[colunas_tipo[0]].value_counts())
    print()

if colunas_operacao:
    print(f"Valores ORIGINAIS na coluna '{colunas_operacao[0]}':")
    print(df_transacao[colunas_operacao[0]].value_counts())
    print()

if colunas_simbolo:
    print(f"Valores ORIGINAIS na coluna '{colunas_simbolo[0]}':")
    print(df_transacao[colunas_simbolo[0]].value_counts())
    print()

print("Primeiras linhas ANTES da tradução:")
df_transacao.head()

Valores ORIGINAIS na coluna 'tipo':
tipo
VYDAJ     634571
PRIJEM    405083
VYBER      16666
Name: count, dtype: int64

Valores ORIGINAIS na coluna 'operacao':
operacao
VYBER             434918
PREVOD NA UCET    208283
VKLAD             156743
PREVOD Z UCTU      65226
VYBER KARTOU        8036
Name: count, dtype: int64

Valores ORIGINAIS na coluna 'simbolo_k':
simbolo_k
UROK           183114
SLUZBY         155832
SIPO           118065
                53433
DUCHOD          30338
POJISTNE        18500
UVER            13580
SANKC. UROK      1577
Name: count, dtype: int64

Primeiras linhas ANTES da tradução:


,identificacao_trans,id_da_conta,data,tipo,operacao,quantia,equilibrio,simbolo_k,banco,conta
0,1,1,1995-03-24,PRIJEM,VKLAD,1000,1000,NaN,NaN,NaN
1,5,1,1995-04-13,PRIJEM,PREVOD Z UCTU,3679,4679,NaN,AB,41403269.0
2,6,1,1995-05-13,PRIJEM,PREVOD Z UCTU,3679,20977,NaN,AB,41403269.0
3,7,1,1995-06-13,PRIJEM,PREVOD Z UCTU,3679,26835,NaN,AB,41403269.0
4,8,1,1995-07-13,PRIJEM,PREVOD Z UCTU,3679,30415,NaN,AB,41403269.0


## Tradução dos valores categóricos

In [25]:
# Traduzir valores das colunas categóricas
colunas_tipo = [col for col in df_transacao.columns if 'tipo' in col.lower() or 'type' in col.lower()]
colunas_operacao = [col for col in df_transacao.columns if 'operacao' in col.lower() or 'operation' in col.lower()]
colunas_simbolo = [col for col in df_transacao.columns if 'simbolo' in col.lower() or 'symbol' in col.lower()]

if colunas_tipo:
    coluna_tipo = colunas_tipo[0]
    df_transacao[coluna_tipo] = traduzir_valores_coluna(df_transacao[coluna_tipo])
    print(f"Valores TRADUZIDOS na coluna '{coluna_tipo}':")
    print(df_transacao[coluna_tipo].value_counts())
    print()

if colunas_operacao:
    coluna_operacao = colunas_operacao[0]
    df_transacao[coluna_operacao] = traduzir_valores_coluna(df_transacao[coluna_operacao])
    print(f"Valores TRADUZIDOS na coluna '{coluna_operacao}':")
    print(df_transacao[coluna_operacao].value_counts())
    print()

if colunas_simbolo:
    coluna_simbolo = colunas_simbolo[0]
    df_transacao[coluna_simbolo] = traduzir_valores_coluna(df_transacao[coluna_simbolo])
    print(f"Valores TRADUZIDOS na coluna '{coluna_simbolo}':")
    print(df_transacao[coluna_simbolo].value_counts())

Valores TRADUZIDOS na coluna 'tipo':
tipo
EMITIR      634571
RECEPCAO    405083
ESCOLHER     16666
Name: count, dtype: int64

Valores TRADUZIDOS na coluna 'operacao':
operacao
ESCOLHER                    434918
TRANSFERENCIA PARA CONTA    208283
DEPOSITO                    156743
TRANSFERENCIA DA UCT         65226
ESCOLHER PELO CARTAO          8036
Name: count, dtype: int64

Valores TRADUZIDOS na coluna 'simbolo_k':
simbolo_k
CHARME                  183114
SERVICOS                155832
SIPO                    118065
                         53433
FANTASMA                 30338
COMPANHIA DE SEGUROS     18500
ACREDITAR                13580
SANKC. UROK               1577
Name: count, dtype: int64


## Visualização DEPOIS da tradução

In [26]:
print("Primeiras linhas DEPOIS da tradução:")
df_transacao.head()

Primeiras linhas DEPOIS da tradução:


,identificacao_trans,id_da_conta,data,tipo,operacao,quantia,equilibrio,simbolo_k,banco,conta
0,1,1,1995-03-24,RECEPCAO,DEPOSITO,1000,1000,NaN,NaN,NaN
1,5,1,1995-04-13,RECEPCAO,TRANSFERENCIA DA UCT,3679,4679,NaN,AB,41403269.0
2,6,1,1995-05-13,RECEPCAO,TRANSFERENCIA DA UCT,3679,20977,NaN,AB,41403269.0
3,7,1,1995-06-13,RECEPCAO,TRANSFERENCIA DA UCT,3679,26835,NaN,AB,41403269.0
4,8,1,1995-07-13,RECEPCAO,TRANSFERENCIA DA UCT,3679,30415,NaN,AB,41403269.0


# Resumo Final

Todas as tabelas foram carregadas e traduzidas com sucesso!

In [27]:
# Resumo de todas as tabelas carregadas
tabelas = {
    'df_conta': df_conta,
    'df_cartao': df_cartao,
    'df_cliente': df_cliente,
    'df_disposicao': df_disposicao,
    'df_distrito': df_distrito,
    'df_emprestimo': df_emprestimo,
    'df_ordem': df_ordem,
    'df_transacao': df_transacao
}

print("\nResumo das Tabelas Carregadas:")
print("=" * 80)
for nome_df, df in tabelas.items():
    print(f"{nome_df:20} | Registros: {df.shape[0]:6} | Colunas: {df.shape[1]:2}")
print("=" * 80)


Resumo das Tabelas Carregadas:
df_conta             | Registros:   4500 | Colunas:  4
df_cartao            | Registros:    892 | Colunas:  4
df_cliente           | Registros:   5369 | Colunas:  4
df_disposicao        | Registros:   5369 | Colunas:  4
df_distrito          | Registros:     77 | Colunas: 16
df_emprestimo        | Registros:    682 | Colunas:  7
df_ordem             | Registros:   6471 | Colunas:  6
df_transacao         | Registros: 1056320 | Colunas: 10


In [28]:
# Fechar conexão
conn.close()
print("Conexão com o banco de dados fechada.")

Conexão com o banco de dados fechada.
